In [1]:
import json, math
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple

In [2]:
import numpy as np
from datasets import Dataset
from tqdm import tqdm
import torch, transformers
from transformers import TrainingArguments, set_seed, BitsAndBytesConfig, TrainerCallback, EarlyStoppingCallback
from transformers import AutoTokenizer, AutoModelForCausalLM
set_seed(42)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [3]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

class TsvLogger(TrainerCallback):
    def __init__(self, path=Path("logs") / "training.tsv"):
        path.parent.mkdir(parents=True, exist_ok=True)
        self.path = path

        columns = ['step', 'epoch', 'split', 'loss', 'learning_rate', 'created_at']
        if self.path.exists():
            return

        with open(self.path, "w", encoding="utf-8") as f:
            f.write('\t'.join(columns) + '\n')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero: # 分布式只在主进程写
            return

        if logs is None or logs.get("loss") is None:
            return

        #print(f"on_log: args={args}, state={state}, control={control}, logs={logs}")

        split = "train"
        step = logs.get("step", state.global_step)
        epoch = max(1, math.ceil(state.epoch))
        loss = round(logs.get("loss", np.nan), 3)        # 训练损失
        lr = round(logs.get("learning_rate", np.nan), 6) 

        values = [str(step), str(epoch), split, str(loss), str(lr), now()]

        with open(self.path, "a", encoding="utf-8") as f:
            f.write('\t'.join(values) + '\n')

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not state.is_world_process_zero:
            return

        if metrics is None or metrics.get("eval_loss") is None:
            return

        #print(f"on_evaluate: args={args}, state={state}, control={control}, metrics={metrics}")

        split = "eval"
        epoch = max(1, math.ceil(state.epoch))
        loss = round(metrics.get("eval_loss", np.nan), 3)
        lr = round(metrics.get("learning_rate", np.nan), 6)

        values = [str(state.global_step), str(epoch), split, str(loss), str(lr), now()]

        with open(self.path, "a", encoding="utf-8") as f:
            f.write('\t'.join(values) + '\n')

In [4]:
run_name = "ch12_sft-full"

ckpt_dir = Path("data") / "ch12" / "checkpoint-500"

data_dir = Path("data") / "ch12" / "sft-full"
data_dir.mkdir(parents=True, exist_ok=True)

In [5]:
#tokenizer = AutoTokenizer.from_pretrained(data_dir / "tokenizer", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
#tokenizer.padding_side = "right"

In [6]:
IGNORE = -100

def chat_into_tokens(messages: List[Dict[str,str]], max_len: int = None):
    last_a = max((i for i, m in enumerate(messages) if m["role"].lower() == "assistant"), default=None)
    if last_a is None:
        raise ValueError("the last message must be an assistant reply")

    # 上下文（包括可能的 system/user/历史 assistant）
    prompt_parts = []
    for i, m in enumerate(messages[:last_a]):
        role = m["role"].strip().lower()
        prompt_parts.append(f"#### {role.title()}: {m['content'].strip()}\n")

    prompt = "".join(prompt_parts) + "\n#### Assistant:"
    answer = " " + messages[last_a]["content"].strip()

    #p = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    p = tokenizer.encode(prompt, add_special_tokens=False)
    a = tokenizer.encode(answer, add_special_tokens=False)

    ids   = p + a + [tokenizer.eos_token_id]
    labels = [-IGNORE] * len(p) + a + [tokenizer.eos_token_id]  # -IGNORE 等价于 -100
    if max_len is not None:   # 可选截断保护
        ids   = ids[:max_len]
        labels = labels[:max_len]

    return ids, labels

def pack_into_blocks(ex_list: List[List[Dict[str,str]]], block_size: int, keep_last: bool = False) -> Dict[str, List[List[int]]]:
    ids_buf, lbl_buf = [], []
    x_list, y_list = [], []

    for msgs in ex_list:
        ids, lbl = chat_into_tokens(msgs)
        ids_buf.extend(ids)
        lbl_buf.extend(lbl)
        # 不断切出整块
        while len(ids_buf) >= block_size:
            x_list.append(ids_buf[:block_size]);
            y_list.append(lbl_buf[:block_size])
            ids_buf = ids_buf[block_size:]
            lbl_buf = lbl_buf[block_size:]
    # 末尾处理
    if keep_last and len(ids_buf) > 0:
        pad_len = block_size - len(ids_buf)

        x_list.append(ids_buf + [tokenizer.pad_token_id]*pad_len)
        y_list.append(lbl_buf + [IGNORE]*pad_len) # label 的 pad 必须是 -100

    return [{"input_ids": v[0], "labels": v[1]} for v in zip(x_list, y_list)]

In [7]:
total_epoches = 5
batch_size = 8
block_size = 1024

In [8]:
with open(Path("data") / "tokenizer" / "train.chats.json", 'r') as f:
    train_raw = json.load(f)

with open(Path("data") / "tokenizer" / "validation.chats.json", 'r') as f:
    eval_raw = json.load(f)

print(f"train_raw: {len(train_raw)}\n{train_raw[0]}")

train_ds = pack_into_blocks(train_raw, block_size=block_size, keep_last=True)
eval_ds = pack_into_blocks(eval_raw, block_size=block_size, keep_last=True)

print(len(train_raw), len(eval_raw))
print(f"Dataset: train={len(train_ds)}, eval={len(eval_ds)}")

train_raw: 20158
[{'message_id': '6ab24d72-0181-4594-a9cd-deaf170242fb', 'parent_id': None, 'content': 'Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.', 'role': 'user'}, {'message_id': 'c8e83833-ecbc-44fe-b6db-735228c25a1c', 'parent_id': '6ab24d72-0181-4594-a9cd-deaf170242fb', 'content': '"Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.\n\nRecent research has identified potential monopsonies in industries such as retai

In [9]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
# bnb = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)

lora_parameters = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.1,
    r=32,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

In [10]:
# model = LlamaForCausalLM.from_pretrained(
base_model = AutoModelForCausalLM.from_pretrained(
    ckpt_dir,
    local_files_only=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",           # 多卡自动放置
    attn_implementation="sdpa",  # 若已安装 flash-attn, 则使用 flash_attention_2；否则删掉或改 "sdpa"

    #quantization_config=bnb,
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

#print("named_modules:", list(base_model.named_modules()))
#for name, module in base_model.named_modules():
#    if any(x in name for x in ["proj", "fc"]):
#        print(name)

Memory footprint: 0.1 GB


In [11]:
ckpts = sorted(
    data_dir.glob("checkpoint-*/"),
    key=lambda x: int(x.name.replace("checkpoint-", "")),
    reverse=True,
)


if len(ckpts) > 0:
    print(f"==> resume_from_checkpoint: {ckpts[0]}")

resume_from_checkpoint = len(ckpts) > 0

==> resume_from_checkpoint: data/ch12/sft-full/checkpoint-2400


In [12]:
# Next, specify the general configuration parameters for training
train_parameters = SFTConfig(
    run_name=run_name,
    output_dir=data_dir,
    num_train_epochs=total_epoches,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=16,
    optim="paged_adamw_32bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    group_by_length=True,
    lr_scheduler_type="reduce_lr_on_plateau",
    report_to="none",
    max_grad_norm=0.3,
    max_steps=5_000,
    save_strategy="steps",
    learning_rate=1e-04,
    weight_decay=0.001,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    #warmup_ratio=0.03,

    #max_seq_length=MAX_SEQUENCE_LENGTH,
    #dataset_text_field="text",
    #hub_strategy="every_save",
    #push_to_hub=True,
    #hub_model_id=HUB_MODEL_ID,
    #hub_private_repo=True,

    completion_only_loss=True,
    restore_callback_states_from_checkpoint=True,
    save_total_limit=5,

    packing=True,
)

callbacks = [
    TsvLogger(data_dir / "training.tsv"),
    EarlyStoppingCallback(early_stopping_patience=10, early_stopping_threshold=1e-03),
]

In [13]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=Dataset.from_list(train_ds),
    eval_dataset=Dataset.from_list(eval_ds),
    args=train_parameters,
    # data_collator=collator, # Don't set data_collator
    callbacks=callbacks,
    # peft_config=lora_parameters,
)

/home/appuser/.local/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:453: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(
/home/appuser/.local/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:495: UserWarning: You are using packing, but the attention implementation is not set to 'flash_attention_2' or 'kernels-community/vllm-flash-attn3'. Packing flattens batches into a single sequence, and Flash Attention is the only known attention mechanisms that reliably support this. Using other implementations may lead t

Packing train dataset:   0%|          | 0/5859 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/298 [00:00<?, ? examples/s]

In [14]:
trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print("Training finished")

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss
2600,4.239700,6.393745


TrainOutput(global_step=2600, training_loss=0.3298308621920072, metrics={'train_runtime': 668.767, 'train_samples_per_second': 956.985, 'train_steps_per_second': 7.476, 'total_flos': 8.4331291213824e+16, 'train_loss': 0.3298308621920072})

In [15]:
# https://huggingface.co/docs/trl/sft_trainer